# recount2 CLAMP models with C2CP prior — Pathway Coverage vs Sample Size

**Environment:** `clamp-analyses`

Reproduces the multi-plier sample-size pathway-coverage analysis (Taroni 2018)
using CLAMP with the C2CP prior.

Follows the same preprocessing pipeline as `00_recount2.ipynb`
(biomaRt mapping → FBM → `preprocessCLAMPFBM` → `zscoreCLAMPFBM`),
then subsamples columns and runs CLAMP for each sample size × seed
instead of training one full model.

Sample sizes: 500, 1000, 2000, 4000, 8000, 16000, 32000 (3 seeds each)

Output: `output/recount2/c2cp_subsample_{N}_seed_{run_idx}/`

## Load libraries

In [32]:
start_time <- Sys.time()
cat("recount2 CLAMP coverage analysis started at:", format(start_time), "\n")

recount2 CLAMP coverage analysis started at: 2026-02-24 16:12:14 


In [33]:
if (!requireNamespace("CLAMP", quietly = TRUE)) {
    REPO_PATH <- "/home/msubirana/Documents/pivlab/CLAMP"
    remotes::install_local(REPO_PATH, force = TRUE, dependencies = FALSE)
}

library(bigstatsr)
library(data.table)
library(dplyr)
library(rsvd)
library(glmnet)
library(Matrix)
library(knitr)
library(here)
library(CLAMP)

source(here("config.R"))

## Configuration

In [34]:
output_data_dir <- file.path(config$GENERAL$OUTPUT_DIR, "recount2")
dir.create(output_data_dir, showWarnings = FALSE, recursive = TRUE)

# C2CP prior path
data_path <- here("data", "archs4")

# Input files (same as 00_recount2.ipynb)
out_dir   <- here("data", "recount2")
rds_file  <- file.path(out_dir, "recount2_PLIER_data", "recount_data_prep_PLIER.RDS")
rpkm_file <- file.path(out_dir, "recount2_PLIER_data", "recount_rpkm.RDS")

# Random seeds (5 replicates per sample size)
base_seed    <- 123
n_replicates <- 5
seeds        <- base_seed + 0:(n_replicates - 1)

# Sample sizes (matching multi-plier paper)
sample_sizes <- c(500, 1000, 2000, 4000, 8000, 16000, 32000)

block_size <- config$GENERAL$CHUNK_SIZE
N_CORES    <- config$recount2$N_CORES

message("Output dir   : ", output_data_dir)
message("Seeds        : ", paste(seeds, collapse = ", "))
message("Sample sizes : ", paste(sample_sizes, collapse = ", "))

Output dir   : /home/msubirana/Documents/pivlab/clamp-analyses/output/recount2

Seeds        : 123, 124, 125, 126, 127

Sample sizes : 500, 1000, 2000, 4000, 8000, 16000, 32000



## Preprocess recount2 data

Identical to `00_recount2.ipynb`: Ensembl→HGNC mapping via biomaRt,
FBM creation, `preprocessCLAMPFBM`, `zscoreCLAMPFBM`.
Skipped on subsequent runs if the filtered FBM already exists.

In [35]:
# Read gene symbols and sample names from the PLIER-prepared RDS
data_prep <- readRDS(rds_file)

meta <- list()
meta$gene_symbols <- rownames(data_prep$rpkm.cm)
meta$samples      <- colnames(data_prep$rpkm.cm)

rm(data_prep)

In [36]:
# Ensembl → HGNC symbol mapping via biomaRt (same as 00_recount2.ipynb)
rpkm.df <- readRDS(rpkm_file)

mart <- biomaRt::useDataset("hsapiens_gene_ensembl",
                            biomaRt::useMart("ensembl"))

rpkm.df$ensembl_gene_id <- unlist(lapply(strsplit(rpkm.df$ENSG, "[.]"), `[[`, 1))

gene.df <- biomaRt::getBM(
    filters    = "ensembl_gene_id",
    attributes = c("ensembl_gene_id", "hgnc_symbol"),
    values     = rpkm.df$ensembl_gene_id,
    mart       = mart
)
gene.df <- gene.df %>% dplyr::filter(complete.cases(.))

rpkm.df <- dplyr::inner_join(gene.df, rpkm.df,
                             by = "ensembl_gene_id",
                             relationship = "many-to-many")

rownames(rpkm.df) <- make.names(rpkm.df$hgnc_symbol, unique = TRUE)
rpkm.df <- rpkm.df %>% dplyr::select(-c(ensembl_gene_id:ENSG))

data_mat <- rpkm.df[meta$gene_symbols, meta$samples]
rm(rpkm.df)

data_mat <- as.matrix(as.data.table(data_mat))

In [37]:
# Remove stale FBM files before recreating (same as 00_recount2.ipynb)
fbm_files <- c(
    file.path(output_data_dir, "FBMrecount2_cov.bk"),
    file.path(output_data_dir, "FBMrecount2_cov_preproc.bk"),
    file.path(output_data_dir, "FBMrecount2_cov_preproc_filtered.bk")
)
for (f in fbm_files) {
    if (file.exists(f)) {
        unlink(f)
        message("Removed stale FBM file: ", f)
    }
}

In [38]:
# Create the FBM
n_genes_raw <- length(meta$gene_symbols)
n_samps_raw <- length(meta$samples)

fbm_file   <- file.path(output_data_dir, "FBMrecount2_cov")
recount2FBM <- FBM(
    nrow        = n_genes_raw,
    ncol        = n_samps_raw,
    backingfile = fbm_file,
    create_bk   = TRUE
)

In [39]:
# Populate FBM in row-blocks
n_blocks <- ceiling(n_genes_raw / block_size)
for (i in seq_len(n_blocks)) {
    start_row <- (i - 1) * block_size + 1
    end_row   <- min(i * block_size, nrow(data_mat))
    recount2FBM[start_row:end_row, ] <- as.matrix(data_mat[start_row:end_row, ])
}
rm(data_mat)
gc()

,used,(Mb),gc trigger,(Mb),max used,(Mb)
Ncells,5986766,319.8,10943481,584.5,10943481,584.5
Vcells,11954231,91.3,3048533548,23258.5,4744704167,36199.3


In [40]:
# Preprocess and z-score FBM
prep_recount2 <- preprocessCLAMPFBM(
    fbm         = recount2FBM,
    mean_cutoff = config$recount2$GENES_MEAN_CUTOFF,
    var_cutoff  = config$recount2$GENES_VAR_CUTOFF
)

recount2_fbm_filt <- prep_recount2$fbm_filtered
recount2_rowStats <- prep_recount2$rowStats

Applying log2 transformation

Filling NAs with 0



In [41]:
zscoreCLAMPFBM(recount2_fbm_filt, recount2_rowStats)

Applying Z-score transformation



In [42]:
samples        <- meta$samples
recount2_genes <- meta$gene_symbols[prep_recount2$kept_rows]

n_genes       <- nrow(recount2_fbm_filt)
n_samps_total <- ncol(recount2_fbm_filt)

message("Preprocessed FBM: ", n_genes, " genes x ", n_samps_total, " samples")

# Save gene list so 02_c2cp_coverage.ipynb can compute the matched pathway count
saveRDS(recount2_genes, file.path(output_data_dir, "recount2_genes.rds"))
saveRDS(samples,        file.path(output_data_dir, "recount2_samples.rds"))

Preprocessed FBM: 6000 genes x 37032 samples



## Load C2CP pathway prior

In [43]:
# Load C2CP pathway
c2_gmt <- CLAMP:::read_gmt(file.path(data_path, "c2.cp.v2026.1.Hs.symbols.gmt"))
names(c2_gmt) <- paste0("C2CP_", names(c2_gmt))
c2_pathMat <- gmtListToSparseMat(list(C2CP = c2_gmt))
C2CP_matched <- getMatchedPathwayMat(c2_pathMat, recount2_genes)
message("Loaded and matched C2CP pathway matrix")

There are 5691 genes in the intersection between data and prior

Removing 1446 pathways

Loaded and matched C2CP pathway matrix



## Run CLAMP models — loop over sample sizes and seeds

In [ ]:
results_summary <- data.frame(
    sample_size = integer(),
    run         = integer(),
    seed        = integer(),
    n_samples   = integer(),
    CLAMP_K     = integer(),
    n_lvs_total = integer(),
    n_lvs_auc70 = integer(),
    n_lvs_auc90 = integer(),
    stringsAsFactors = FALSE
)

message("Total samples in preprocessed FBM: ", n_samps_total)

for (n_target in sample_sizes) {
    for (run_idx in seq_len(n_replicates)) {

        current_seed <- seeds[run_idx]

        message("\n", strrep("=", 60))
        message("SAMPLE SIZE: ", n_target,
                " | RUN ", run_idx, "/", n_replicates,
                " | SEED: ", current_seed)
        message(strrep("=", 60))

        output_dir <- file.path(
            output_data_dir,
            paste0("c2cp_subsample_", n_target, "_seed_", run_idx)
        )
        dir.create(output_dir, showWarnings = FALSE, recursive = TRUE)

        # Skip if already complete
        if (file.exists(file.path(output_dir, "CLAMPfull_C2CP.rds"))) {
            message("Output already exists — skipping.")
            next
        }

        # ── Sample selection ──────────────────────────────────────────────
        set.seed(current_seed)
        n_draw     <- min(n_target, n_samps_total)
        sample_idx <- sort(sample(seq_len(n_samps_total), n_draw))
        n_samples  <- length(sample_idx)
        samp_names <- samples[sample_idx]

        message("Selected ", n_samples, " samples")

        saveRDS(
            list(
                sample_size     = n_target,
                run             = run_idx,
                seed            = current_seed,
                n_samples       = n_samples,
                sample_idx      = sample_idx,
                sample_names    = samp_names,
                sampling_method = "random_sampling"
            ),
            file = file.path(output_dir, "subsample_info.rds")
        )

        # ── Subsampled FBM ────────────────────────────────────────────────
        message("Creating subsampled FBM...")
        fbm_sub_file <- file.path(output_dir, "fbm_subsampled")
        Y_sub <- big_copy(
            recount2_fbm_filt,
            ind.col     = sample_idx,
            backingfile = fbm_sub_file
        )

        # ── SVD ───────────────────────────────────────────────────────────
        message("Computing SVD...")
        SVD_K <- min(n_samples - 1, n_genes - 1)

        if (N_CORES > 1) {
            options(bigstatsr.check.parallel.blas = FALSE)
            blas_nproc <- getOption("default.nproc.blas")
            options(default.nproc.blas = NULL)
        }

        svd_result <- big_randomSVD(Y_sub, k = SVD_K, ncores = N_CORES)

        if (N_CORES > 1) {
            options(bigstatsr.check.parallel.blas = TRUE)
            options(default.nproc.blas = blas_nproc)
        }

        valid_idx    <- which(!is.nan(svd_result$d))
        svd_result$d <- svd_result$d[valid_idx]
        svd_result$u <- svd_result$u[, valid_idx, drop = FALSE]
        svd_result$v <- svd_result$v[, valid_idx, drop = FALSE]

        saveRDS(svd_result, file.path(output_dir, "svd.rds"))

        # ── Estimate CLAMP K ──────────────────────────────────────────────
        CLAMP_K <- num.pc(list(d = svd_result$d)) * 2
        message("CLAMP K = ", CLAMP_K)
        saveRDS(CLAMP_K, file.path(output_dir, "CLAMP_K.rds"))

        # ── CLAMPbase ─────────────────────────────────────────────────────
        message("Running CLAMPbase...")
        baseRes <- CLAMPbase(
            Y       = Y_sub,
            svdres  = svd_result,
            trace   = TRUE,
            clamp_k = CLAMP_K
        )

        baseRes$Z <- data.frame(baseRes$Z)
        rownames(baseRes$Z) <- recount2_genes
        baseRes$B <- data.frame(baseRes$B)
        colnames(baseRes$B) <- samp_names

        saveRDS(baseRes, file.path(output_dir, "CLAMPbase.rds"))

        model_dir <- file.path(output_dir, "CLAMPbase")
        dir.create(model_dir, showWarnings = FALSE, recursive = TRUE)
        write.csv(baseRes$B, file.path(model_dir, "B.csv"))
        write.csv(baseRes$Z, file.path(model_dir, "Z.csv"))

        # ── CLAMPfull with C2CP prior ─────────────────────────────────────
        message("Running CLAMPfull with C2CP prior...")
        fullRes <- CLAMPfull(
            Y                 = Y_sub,
            svdres            = svd_result,
            priorMat          = C2CP_matched,
            clamp.base.result = baseRes,
            use_cpp           = TRUE,
            trace             = TRUE,
            clamp_k           = CLAMP_K
        )

        fullRes$Z <- data.frame(fullRes$Z)
        rownames(fullRes$Z) <- recount2_genes
        fullRes$B <- data.frame(fullRes$B)
        colnames(fullRes$B) <- samp_names
        fullRes$summary <- fullRes$summary %>%
            dplyr::rename(LV = LV_index) %>%
            dplyr::mutate(LV = paste0("LV", LV))

        saveRDS(fullRes, file.path(output_dir, "CLAMPfull_C2CP.rds"))

        model_dir <- file.path(output_dir, "CLAMPfull_C2CP")
        dir.create(model_dir, showWarnings = FALSE, recursive = TRUE)
        write.csv(fullRes$B,       file.path(model_dir, "B.csv"))
        write.csv(fullRes$Z,       file.path(model_dir, "Z.csv"))
        write.csv(fullRes$summary, file.path(model_dir, "summary.csv"))

        # ── Record summary ────────────────────────────────────────────────
        n_lvs_total <- nrow(fullRes$summary)
        n_lvs_auc70 <- sum(fullRes$summary$AUC >= 0.70, na.rm = TRUE)
        n_lvs_auc90 <- sum(fullRes$summary$AUC >= 0.90, na.rm = TRUE)

        results_summary <- rbind(results_summary, data.frame(
            sample_size = n_target,
            run         = run_idx,
            seed        = current_seed,
            n_samples   = n_samples,
            CLAMP_K     = CLAMP_K,
            n_lvs_total = n_lvs_total,
            n_lvs_auc70 = n_lvs_auc70,
            n_lvs_auc90 = n_lvs_auc90
        ))

        # ── Clean up memory and subsampled FBM ───────────────────────────
        rm(Y_sub, svd_result, baseRes, fullRes)
        gc()
    }
}

message("\nAll runs completed!")

Total samples in preprocessed FBM: 37032



SAMPLE SIZE: 500 | RUN 1/5 | SEED: 123


Selected 500 samples

Creating subsampled FBM...

Computing SVD...

CLAMP K = 20

Running CLAMPbase...

****

CLAMP k is set to 20

L1 is set to 40.750078028982

L2 is set to 122.250234086946

Progress 1 / 200 | Bdiff=0.476176, minCor=0.890887

Progress 2 / 200 | Bdiff=0.029735, minCor=0.950349

Progress 3 / 200 | Bdiff=0.010197, minCor=0.990342

Progress 4 / 200 | Bdiff=0.005196, minCor=0.993062

Progress 5 / 200 | Bdiff=0.003373, minCor=0.994636

Progress 6 / 200 | Bdiff=0.002570, minCor=0.995537

Progress 7 / 200 | Bdiff=0.002147, minCor=0.995579

Progress 8 / 200 | Bdiff=0.001883, minCor=0.995837

Progress 9 / 200 | Bdiff=0.001682, minCor=0.996303

Progress 10 / 200 | Bdiff=0.001512, minCor=0.996854

Progress 11 / 200 | Bdiff=0.001349, minCor=0.997415

Progress 12 / 200 | Bdiff=0.001197, minCor=0.997452

Progress 13 / 200 | Bdiff=0.001063, minCor=0.997273

Progress 14 / 200 | Bdiff=0.000951, minCor

## Save and display results summary

In [ ]:
saveRDS(
    results_summary,
    file.path(output_data_dir, "subsample_results_summary.rds")
)
write.csv(
    results_summary,
    file.path(output_data_dir, "subsample_results_summary.csv"),
    row.names = FALSE
)

print(results_summary)

In [ ]:
end_time     <- Sys.time()
elapsed_time <- end_time - start_time
cat("Analysis completed at:", format(end_time), "\n")
cat("Total elapsed time   :", format(elapsed_time), "\n")